In [1]:
!pip install datasets tensorflow scikit-learn

In [2]:
import re
import numpy as np

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

from sklearn.model_selection import train_test_split

In [4]:
dataset = load_dataset("wangrongsheng/ag_news")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [5]:
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

train_texts = [clean_text(text) for text in train_texts]
test_texts = [clean_text(text) for text in test_texts]

In [7]:
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(train_texts)

X_train = tokenizer.texts_to_sequences(train_texts)

X_test = tokenizer.texts_to_sequences(test_texts)

In [8]:
max_length = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [12]:
max_length = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)
print(X_train.shape)


(120000, 50)


In [13]:
y_train = to_categorical(train_labels, num_classes=4)

y_test = to_categorical(test_labels, num_classes=4)

In [14]:
model = Sequential()

model.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_length
    )
)

model.add(
    SimpleRNN(64)
)

model.add(
    Dense(4, activation="softmax")
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 45s 29ms/step - accuracy: 0.8026 - loss: 0.5507 - val_accuracy: 0.8603 - val_loss: 0.4049
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 44s 29ms/step - accuracy: 0.8976 - loss: 0.3262 - val_accuracy: 0.8804 - val_loss: 0.3666
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 81s 28ms/step - accuracy: 0.9202 - loss: 0.2599 - val_accuracy: 0.8829 - val_loss: 0.3650
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 44s 30ms/step - accuracy: 0.9321 - loss: 0.2217 - val_accuracy: 0.8779 - val_loss: 0.3931
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 81s 29ms/step - accuracy: 0.9418 - loss: 0.1913 - val_accuracy: 0.8656 - val_loss: 0.4248


In [18]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8766 - loss: 0.3857
Test Loss : 0.3857189118862152
Test Accuracy : 0.8765789270401001


In [19]:
sample = [
    "India wins the cricket world cup after defeating Australia."
]

sample = [clean_text(text) for text in sample]

sample = tokenizer.texts_to_sequences(sample)

sample = pad_sequences(
    sample,
    maxlen=max_length,
    padding='post'
)

prediction = model.predict(sample)

label = np.argmax(prediction)

classes = [
    "World",
    "Sports",
    "Business",
    "Science/Technology"
]

print("Predicted Class :", classes[label])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
Predicted Class : Sports
